In [109]:
# Check if autoreload is loaded and load/reload accordingly
try:
    %reload_ext autoreload
except:
    %load_ext autoreload
%autoreload 2
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [110]:
forward_rates = ['k_{01}', 'k_{12}', 'k_{23}', 'k_{30}']
backward_rates = ['k_{10}', 'k_{21}', 'k_{32}']

config_params = yaml.safe_load(Path("../test_files/random_ring_models/base_4ring_rate_params.yaml").open())
rate_params = config_params['Rates']

for rate_name in forward_rates:
    rate_dict = next(item for item in rate_params if item["name"] == rate_name)
    base_rate = np.random.uniform(.5, 5)
    mut_rate = np.random.uniform(.1, base_rate) # Mutated rate must be slower than base rate
    rate_dict['rate_vals'] = [base_rate, mut_rate]
    rate_dict['weight_distr'] = f"single_free_energy_mat_from_kinetic_rates(rate_of_species=([{base_rate}, {mut_rate}]))"

for rate_name in backward_rates:
    rate_dict = next(item for item in rate_params if item["name"] == rate_name)
    # We might not need to enforce this
    reciprical_rate = next(item for item in rate_params if item["state_list"] == rate_dict["state_list"][::-1])
    base_rate = np.random.uniform(0.01, reciprical_rate['rate_vals'][0])  # Backward rate must be slower than forward rate
    mut_rate = base_rate + np.random.uniform(.1, 5.0) # Mutated rate must be faster than base rate
    rate_dict['rate_vals'] = [base_rate, mut_rate]
    rate_dict['weight_distr'] = f"single_free_energy_mat_from_kinetic_rates(rate_of_species=([{base_rate}, {mut_rate}]))"

print(config_params)
yaml.safe_dump(config_params, Path("../test_files/random_ring_models/generated_4ring_rate_params.yaml").open('w'))

{'Title': 'test_4ring', 'Input': {'seq_length': 7, 'template': 'AAAAAAA', 'values': ['A', 'B']}, 'States': ['0', '1', '2', '3'], 'Rates': [{'name': 'k_{01}', 'state_list': ['0', '1'], 'input_range': [0, 1], 'weight_distr': 'single_free_energy_mat_from_kinetic_rates(rate_of_species=([1.4645560577056347, 0.5265441268622334]))', 'rate_vals': [1.4645560577056347, 0.5265441268622334]}, {'name': 'k_{10}', 'state_list': ['1', '0'], 'input_range': [1, 2], 'weight_distr': 'single_free_energy_mat_from_kinetic_rates(rate_of_species=([1.192486408634931, 5.215181593861036]))', 'rate_vals': [1.192486408634931, 5.215181593861036]}, {'name': 'k_{12}', 'state_list': ['1', '2'], 'input_range': [2, 3], 'weight_distr': 'single_free_energy_mat_from_kinetic_rates(rate_of_species=([1.1973487899732174, 0.9546143957984158]))', 'rate_vals': [1.1973487899732174, 0.9546143957984158]}, {'name': 'k_{21}', 'state_list': ['2', '1'], 'input_range': [3, 4], 'weight_distr': 'single_free_energy_mat_from_kinetic_rates(rat

In [132]:
from elektrum.king_altman_kinetic_model import  KingAltmanKineticModel
# yaml_file = "/Users/alamson/projects/Elektrum/test_files/test_4ring/test_4ring_rate_params.yaml"
yaml_file = "/Users/alamson/projects/Elektrum/test_files/random_ring_models/generated_4ring_rate_params.yaml"
print(f"Loading model from {yaml_file}")
model = KingAltmanKineticModel(yaml_file)
# print(model.lab_enc.transform(list(model.template)))
# print(model.template)
kinn_mat = model.get_kinetic_mat_for_seq(model.template)
# Get eigenvectors from 
eigenvalues, eigenvectors = np.linalg.eig(kinn_mat)
# Eigen vectors from the value closest to zero eigenvalue
closest_to_zero_idx = np.argmin(np.abs(eigenvalues))
steady_state_vector = eigenvectors[:, closest_to_zero_idx]
steady_state_vector = steady_state_vector / np.sum(steady_state_vector)  # Normalize
# Compute activity using steady state vector

seq_ohe = model.generate_ohe_from_seq(model.template)
print(seq_ohe)

eig_activity = steady_state_vector[3] * model.rates[-1].get_rate(seq_ohe) 


# Use the model
# if model.config.template:
activity = model.get_activity(model.template)
cls_activity = model.get_eigen_activity(model.template)
mut_template = list(model.template)
mut_template[0] = 'B'
mut_template[4] = 'B'
mut_template = ''.join(mut_template)
print(f"Template activity: {activity}")
print(f"Eigen activity: {eig_activity}")
print(f"Class eigen activity: {cls_activity}")
print(f"Mutant template activity: {model.get_activity(mut_template)}")
print(f"Class eigen Mutant template activity: {model.get_eigen_activity(mut_template)}")


Loading model from /Users/alamson/projects/Elektrum/test_files/random_ring_models/generated_4ring_rate_params.yaml
[[1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]
 [1. 0.]]
Template activity: 0.27095805487967506
Eigen activity: (0.2709580548796751+0j)
Class eigen activity: (0.2709580548796751+0j)
Mutant template activity: 0.1375827147365106
Class eigen Mutant template activity: 0.13758271473651074


In [147]:
from elektrum.king_altman_kinetic_model import  KingAltmanKineticModel
# yaml_file = "/Users/alamson/projects/Elektrum/test_files/test_4ring/test_4ring_rate_params.yaml"
yaml_file = "/Users/alamson/projects/Elektrum/test_files/random_ring_models/generated_4ring_rate_params.yaml"
print(f"Loading model from {yaml_file}")
model = KingAltmanKineticModel(yaml_file)
model.gen_simulated_data(mut_num=[1,2,3,4], npoints=1000, save_hdf5=True)

Loading model from /Users/alamson/projects/Elektrum/test_files/random_ring_models/generated_4ring_rate_params.yaml


In [157]:
forward_rates = ['k_{01}', 'k_{12}', 'k_{23}', 'k_{30}']
backward_rates = ['k_{10}', 'k_{21}', 'k_{32}']

config_params = yaml.safe_load(Path("../test_files/random_ring_models/base_4ring_rate_params.yaml").open())
rate_params = config_params['Rates']
# set random seed for reproducibility
np.random.seed(42)

for i in range(100):
    new_config_params = config_params.copy()
    rate_params = new_config_params['Rates']
    for rate_name in forward_rates:
        rate_dict = next(item for item in rate_params if item["name"] == rate_name)
        base_rate = np.random.uniform(.1, 10)
        mut_rate = np.random.uniform(.1, base_rate) # Mutated rate must be slower than base rate
        rate_dict['rate_vals'] = [base_rate, mut_rate]
        rate_dict['weight_distr'] = f"single_free_energy_mat_from_kinetic_rates(rate_of_species=([{base_rate}, {mut_rate}]))"

    for rate_name in backward_rates:
        rate_dict = next(item for item in rate_params if item["name"] == rate_name)
        # We might not need to enforce this
        reciprical_rate = next(item for item in rate_params if item["state_list"] == rate_dict["state_list"][::-1])
        base_rate = np.random.uniform(0.01, reciprical_rate['rate_vals'][0])  # Backward rate must be slower than forward rate
        mut_rate = base_rate + np.random.uniform(.1, 10.0) # Mutated rate must be faster than base rate
        rate_dict['rate_vals'] = [base_rate, mut_rate]
        rate_dict['weight_distr'] = f"single_free_energy_mat_from_kinetic_rates(rate_of_species=([{base_rate}, {mut_rate}]))"
    
    new_config_params['Title'] = f"ring_datasets/test_4ring_random_{i}"
    model = KingAltmanKineticModel(new_config_params)
    model.gen_simulated_data(mut_num=[1,2,3,4,5,6], npoints=500, rng_seed=i, save_hdf5=True)

# print(config_params)
# yaml.safe_dump(config_params, Path("../test_files/random_ring_models/generated_4ring_rate_params.yaml").open('w'))

In [159]:
forward_rates = ['k_{01}', 'k_{12}', 'k_{23}', 'k_{30}']
backward_rates = ['k_{10}', 'k_{21}', 'k_{32}', 'k_{20}']

config_params = yaml.safe_load(Path("../test_files/random_4PR_models/base_4proofreading_params.yaml").open())
rate_params = config_params['Rates']
# set random seed for reproducibility
np.random.seed(42)

for i in range(100):
    new_config_params = config_params.copy()
    rate_params = new_config_params['Rates']
    for rate_name in forward_rates:
        rate_dict = next(item for item in rate_params if item["name"] == rate_name)
        base_rate = np.random.uniform(.1, 10)
        mut_rate = np.random.uniform(.1, base_rate) # Mutated rate must be slower than base rate
        rate_dict['rate_vals'] = [base_rate, mut_rate]
        rate_dict['weight_distr'] = f"single_free_energy_mat_from_kinetic_rates(rate_of_species=([{base_rate}, {mut_rate}]))"

    for rate_name in backward_rates:
        rate_dict = next(item for item in rate_params if item["name"] == rate_name)
        # We might not need to enforce this
        reciprocal_rate = next(
            (item for item in rate_params if item["state_list"] == rate_dict["state_list"][::-1]),
            None
        )
        base_rate = (
            np.random.uniform(0.01, reciprocal_rate['rate_vals'][0])
            if reciprocal_rate
            else np.random.uniform(0.01, 2.0)
        )
        mut_rate = base_rate + np.random.uniform(.1, 10.0) # Mutated rate must be faster than base rate
        rate_dict['rate_vals'] = [base_rate, mut_rate]
        rate_dict['weight_distr'] = f"single_free_energy_mat_from_kinetic_rates(rate_of_species=([{base_rate}, {mut_rate}]))"

    new_config_params['Title'] = f"proofreading_datasets/test_4proofreading_random_{i}"
    model = KingAltmanKineticModel(new_config_params)
    model.gen_simulated_data(mut_num=[1,2,3,4,5,6,7], npoints=500, rng_seed=i, save_hdf5=True)

# print(config_params)
# yaml.safe_dump(config_params, Path("../test_files/random_ring_models/generated_4ring_rate_params.yaml").open('w'))